# Step 1: Explore the LLM before building any app

Goal of this notebook: just talk to the model directly and see what comes back.
No Flask, no FastAPI, no Docker — none of that yet.

Run each cell **one at a time** with Shift+Enter, and read the output before moving to the next.

## Step 1.1: Install what we need
Run this once. If it's already installed, it'll just say 'Requirement already satisfied'.

In [1]:
!pip3 install -r requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip3 install groq python-dotenv


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1.2: Set your API key

Get a free key from https://console.groq.com if you don't have one yet.

**Do not paste your real key directly in a cell that you might commit to git.**
Instead, create a file called `.env` in this same folder with one line:

```
GROQ_API_KEY=your_real_key_here
```

Then run the next cell — it loads that file for you.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = "*"
 
key = os.environ.get("GROQ_API_KEY")
print("Key loaded:", bool(key))  # should print True. If False, check your .env file.

Key loaded: True


In [6]:
!pip3 install "httpx==0.27.2"

  Obtaining dependency information for httpx==0.27.2 from https://files.pythonhosted.org/packages/56/95/9377bcb415797e44274b51d46e3249eba641711cf3348050f76ee7b15ffc/httpx-0.27.2-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/76.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/76.4 kB ? eta -:--:--
   ---------------- ----------------------- 30.7/76.4 kB ? eta -:--:--
   ---------------------------------------- 76.4/76.4 kB 1.4 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip show groq httpx

Name: groq
Version: 0.9.0
Summary: The official Python library for the groq API
Home-page: 
Author: 
Author-email: Groq <support@groq.com>
License: 
Location: c:\Users\niharikamanav\Downloads\gen_ai_training\week6\.venv_docker\Lib\site-packages
Requires: anyio, distro, httpx, pydantic, sniffio, typing-extensions
Required-by: 
---
Name: httpx
Version: 0.27.2
Summary: The next generation HTTP client.
Home-page: 
Author: 
Author-email: Tom Christie <tom@tomchristie.com>
License: 
Location: c:\Users\niharikamanav\Downloads\gen_ai_training\week6\.venv_docker\Lib\site-packages
Requires: anyio, certifi, httpcore, idna, sniffio
Required-by: groq
Note: you may need to restart the kernel to use updated packages.


## Step 1.3: Make the client and send ONE test message
This is the simplest possible call — just to prove the connection works.

In [2]:
from groq import Groq

client = Groq(api_key=key)
MODEL = "llama-3.3-70b-versatile"

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello in one short sentence."}
    ]
)

print(response.choices[0].message.content)

Hello, it's nice to meet you.


Did you see a real reply above? Great — the model connection works. If you got an error, read it carefully: usually it's either a missing/invalid key, or a typo in the model name.

## Step 1.4: Try a couple more messages, one cell at a time
Just change the text below and re-run to get a feel for how the model responds.

In [3]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)
print(response.choices[0].message.content)

The capital of France is Paris.


In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Explain XGBoost to a beginner in 3 sentences."}
    ]
)
print(response.choices[0].message.content)

XGBoost (Extreme Gradient Boosting) is a type of machine learning algorithm that uses a technique called gradient boosting to create highly accurate models by combining multiple weaker models into a stronger one. It works by training a series of decision trees, with each subsequent tree attempting to correct the errors of the previous one, resulting in a highly accurate prediction model. XGBoost is particularly useful for handling large datasets and complex relationships between variables, making it a popular choice in data science competitions and real-world applications.


## Step 1.5: Add a system prompt (controls the model's overall behaviour)
So far we only sent a `user` message. A `system` message sets the tone/rules for every reply.

In [5]:
SYSTEM_PROMPT = "You are a helpful, concise assistant. Keep answers short unless asked for detail."

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "Tell me about the Eiffel Tower."}
    ]
)
print(response.choices[0].message.content)

The Eiffel Tower is a famous iron lattice tower in Paris, France, built for the 1889 World's Fair. It stands 324 meters (1,063 feet) tall and was the world's tallest structure when completed. It's now a iconic symbol of Paris and one of the most visited landmarks globally.


## Step 1.6: Wrap it into a single reusable function
Now that we've seen it work a few times, let's wrap the logic into one function —
this is exactly what will become `chatbot.py` in the next step.

In [6]:
def get_response(user_message: str) -> str:
    if not user_message or not user_message.strip():
        return "Please enter a message."
    try:
        completion = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
            ],
            temperature=0.7,
            max_tokens=512,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error calling LLM: {e}"

# quick test
print(get_response("What's a good name for a data science portfolio project?"))

"Predicting [Outcome] with [Dataset/Method]" or "[Industry/Topic] Data Analysis" are good starting points. For example: "Predicting House Prices with Kaggle Data" or "COVID-19 Case Study: Data Visualization and Insights".


In [7]:
# Try your own question here
print(get_response("Type any question you like here"))

What is the most interesting thing you've learned recently?


## Checkpoint

If you're happy with how `get_response()` behaves above:
- Same idea has already been saved as a plain script in `chatbot.py`
- Next step: open `chatbot.py` in VS Code and run it directly (Step 2 in `STEPS.md`)
- After that, we move to the Flask app (Step 3)

No Docker, no AWS yet — that's a later step once you're confident the app works.